# Pipeline final Taller 2 - Freesound Audio Tagging

Este notebook reconstruye la solucion final del Proyecto 2 desde los datos y artefactos del repositorio hasta el archivo `submission.csv` de Kaggle. El modelo elegido es el blend 3-way e100 reponderado: una rama CNN separable con cabeza densa, una rama temporal con normalizacion global mel y una rama temporal con ventana de 1024 frames.

El modo por defecto no reentrena: recompone y valida el CSV oficial ya scoreado en Kaggle. Para una reproduccion pesada se puede activar `RUN_HEAVY_STEPS=True`, lo que reconstruye caches log-mel y entrena las tres ramas durante 100 epocas.


## Librerias utilizadas

El pipeline usa `pandas` para validar y recomponer submissions, `subprocess` para llamar a los scripts de entrenamiento, y las utilidades ya versionadas en `investigation/scripts/`. El entrenamiento pesado depende de PyTorch, librosa y CUDA cuando hay GPU disponible.


In [1]:
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


## Configuracion fija del experimento final

Esta celda fija las decisiones que definen la entrega: semilla, pesos del blend, rutas de componentes y score privado asociado al CSV oficial. El flag `RUN_HEAVY_STEPS` separa el modo de entrega liviano del modo de reproduccion completa.


In [2]:
RUN_HEAVY_STEPS = False
SEED = 42
EXPECTED_ROWS = 3361
EXPECTED_COLUMNS = 81
EXPECTED_PRIVATE_LB = 0.67126
EXPECTED_SHA256 = '4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab'

ROOT = Path.cwd().resolve()
while not (ROOT / 'data' / 'sample_submission.csv').exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('No se encontro la raiz del repositorio')
    ROOT = ROOT.parent

CODE_DIR = ROOT / 'proyecto_actual' / 'codigo'
DATA_DIR = ROOT / 'data'
INVESTIGATION = ROOT / 'investigation'
SCRIPTS = INVESTIGATION / 'scripts'
OUTPUT_DIR = CODE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = ROOT / '.venv' / 'bin' / 'python'
PYTHON = VENV_PYTHON if VENV_PYTHON.exists() else Path(sys.executable)

COMPONENTS = [
    {
        'id': 'separable_headsep_e100_seed42',
        'short': 'headsep',
        'weight': 0.25,
        'input': 'log-mel 128 x 512',
        'normalization': 'por clip',
        'architecture': 'CNN separable-residual + cabeza densa',
        'role': 'patrones locales tiempo-frecuencia',
        'official_csv': ROOT / 'investigation/submissions/parallel100_20260702_separable_headsep_e100_seed42/small_logmel_cnn.csv',
        'heavy_csv': OUTPUT_DIR / 'separable_headsep_e100' / 'submissions' / 'small_logmel_cnn.csv',
    },
    {
        'id': 'globalmel_sep_temporal_e100_seed42',
        'short': 'globalmel',
        'weight': 0.375,
        'input': 'log-mel 128 x 512',
        'normalization': 'global por banda mel',
        'architecture': 'CNN separable-residual + BiGRU',
        'role': 'escala estable y evolucion temporal',
        'official_csv': ROOT / 'investigation/submissions/parallel100_20260702_globalmel_sep_temporal_e100_seed42/small_logmel_cnn.csv',
        'heavy_csv': OUTPUT_DIR / 'globalmel_e100' / 'submissions' / 'small_logmel_cnn.csv',
    },
    {
        'id': 'sep_temporal_f1024_e100_seed42',
        'short': 'f1024',
        'weight': 0.375,
        'input': 'log-mel 128 x 1024',
        'normalization': 'por clip',
        'architecture': 'CNN separable-residual + BiGRU',
        'role': 'contexto temporal largo',
        'official_csv': ROOT / 'investigation/submissions/parallel100_20260702_sep_temporal_f1024_e100_seed42/small_logmel_cnn.csv',
        'heavy_csv': OUTPUT_DIR / 'f1024_e100' / 'submissions' / 'small_logmel_cnn.csv',
    },
]

FINAL_SUBMISSION = CODE_DIR / 'submission.csv'
METADATA_PATH = CODE_DIR / 'pipeline_final_taller_2_metadata.json'

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def run_command(args: list[str | Path], *, heavy: bool = False) -> None:
    rendered = ' '.join(str(arg) for arg in args)
    print(f'$ {rendered}')
    if heavy and not RUN_HEAVY_STEPS:
        print('SKIP: RUN_HEAVY_STEPS=False')
        return
    subprocess.run([str(arg) for arg in args], check=True, cwd=ROOT)

config_summary = {
    'repo_root': str(ROOT),
    'python_for_subprocesses': str(PYTHON),
    'run_heavy_steps': RUN_HEAVY_STEPS,
    'expected_private_lb': EXPECTED_PRIVATE_LB,
    'expected_sha256': EXPECTED_SHA256,
}
config_summary


{'repo_root': '/home/santig14/fing/taa/2_TallerAA',
 'python_for_subprocesses': '/home/santig14/fing/taa/2_TallerAA/.venv/bin/python',
 'run_heavy_steps': False,
 'expected_private_lb': 0.67126,
 'expected_sha256': '4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab'}

## Datos esperados y metrica

La competencia es multi-label: cada audio puede tener una o mas etiquetas entre 80 clases. El archivo de salida debe respetar exactamente el orden y las columnas de `sample_submission.csv`. La metrica usada por Kaggle es lwlrap, que evalua la calidad del ranking de clases relevantes por audio.


In [3]:
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
train_curated = pd.read_csv(DATA_DIR / 'train_curated.csv')
train_noisy = pd.read_csv(DATA_DIR / 'train_noisy.csv')
label_columns = list(sample_submission.columns[1:])

assert len(label_columns) == 80
assert len(sample_submission) == EXPECTED_ROWS
assert len(sample_submission.columns) == EXPECTED_COLUMNS

dataset_summary = pd.DataFrame([
    {
        'split': 'train_curated',
        'rows': len(train_curated),
        'unique_files': train_curated['fname'].nunique(),
        'labels_per_audio_mean': train_curated['labels'].str.split(',').map(len).mean(),
    },
    {
        'split': 'train_noisy',
        'rows': len(train_noisy),
        'unique_files': train_noisy['fname'].nunique(),
        'labels_per_audio_mean': train_noisy['labels'].str.split(',').map(len).mean(),
    },
    {
        'split': 'test',
        'rows': len(sample_submission),
        'unique_files': sample_submission['fname'].nunique(),
        'labels_per_audio_mean': None,
    },
])
display(dataset_summary)


,split,rows,unique_files,labels_per_audio_mean
0,train_curated,4970,4970,1.157344
1,train_noisy,19815,19815,1.211204
2,test,3361,3361,NaN


## Preprocesamiento comun

Las tres ramas parten de la misma idea: convertir cada `.wav` a mono, remuestrear a 44.1 kHz, calcular un espectrograma Mel, pasar a escala logaritmica y recortar o rellenar el eje temporal. Esa representacion conserva una imagen tiempo-frecuencia que una CNN puede explotar mejor que estadisticas globales.


In [4]:
cache_commands = [
    [PYTHON, SCRIPTS / 'build_logmel_image_cache.py', '--data-dir', DATA_DIR, '--n-mels', '128', '--frames', '512', '--normalization', 'clip'],
    [PYTHON, SCRIPTS / 'build_logmel_image_cache.py', '--data-dir', DATA_DIR, '--n-mels', '128', '--frames', '512', '--normalization', 'global-mel', '--cache-tag', 'globalmel'],
    [PYTHON, SCRIPTS / 'build_logmel_image_cache.py', '--data-dir', DATA_DIR, '--n-mels', '128', '--frames', '1024', '--normalization', 'clip'],
]

for command in cache_commands:
    run_command(command, heavy=True)

cache_summary = pd.DataFrame([
    {'cache': 'curated_logmel_image_m128_f512.npz', 'frames': 512, 'normalization': 'por clip', 'exists': (DATA_DIR / 'curated_logmel_image_m128_f512.npz').exists()},
    {'cache': 'curated_logmel_image_m128_f512_globalmel.npz', 'frames': 512, 'normalization': 'global mel', 'exists': (DATA_DIR / 'curated_logmel_image_m128_f512_globalmel.npz').exists()},
    {'cache': 'curated_logmel_image_m128_f1024.npz', 'frames': 1024, 'normalization': 'por clip', 'exists': (DATA_DIR / 'curated_logmel_image_m128_f1024.npz').exists()},
])
display(cache_summary)


$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --n-mels 128 --frames 512 --normalization clip
SKIP: RUN_HEAVY_STEPS=False
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --n-mels 128 --frames 512 --normalization global-mel --cache-tag globalmel
SKIP: RUN_HEAVY_STEPS=False
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --n-mels 128 --frames 1024 --normalization clip
SKIP: RUN_HEAVY_STEPS=False


,cache,frames,normalization,exists
0,curated_logmel_image_m128_f512.npz,512,por clip,True
1,curated_logmel_image_m128_f512_globalmel.npz,512,global mel,True
2,curated_logmel_image_m128_f1024.npz,1024,por clip,True


## Modelo final

El modelo final no es una arquitectura unica, sino un blend simple de tres ramas complementarias. La primera captura patrones locales tiempo-frecuencia; las otras dos agregan lectura temporal explicita con BiGRU, una con normalizacion global por banda Mel y otra con contexto temporal mas largo.


In [5]:
component_table = pd.DataFrame([
    {
        'component': component['id'],
        'weight': component['weight'],
        'input': component['input'],
        'normalization': component['normalization'],
        'architecture': component['architecture'],
        'role': component['role'],
        'official_exists': component['official_csv'].exists(),
    }
    for component in COMPONENTS
])
assert component_table['official_exists'].all()
assert abs(component_table['weight'].sum() - 1.0) < 1e-12
display(component_table)


,component,weight,input,normalization,architecture,role,official_exists
0,separable_headsep_e100_seed42,0.250,log-mel 128 x 512,por clip,CNN separable-residual + cabeza densa,patrones locales tiempo-frecuencia,True
1,globalmel_sep_temporal_e100_seed42,0.375,log-mel 128 x 512,global por banda mel,CNN separable-residual + BiGRU,escala estable y evolucion temporal,True
2,sep_temporal_f1024_e100_seed42,0.375,log-mel 128 x 1024,por clip,CNN separable-residual + BiGRU,contexto temporal largo,True


## Entrenamiento opcional de las tres ramas

Esta celda contiene los comandos pesados de reproduccion. En modo entrega quedan salteados para que el notebook sea ejecutable en una maquina comun; en modo pesado entrenan 100 epocas con la misma configuracion documentada para el modelo final.


In [6]:
training_commands = {
    'separable_headsep_e100_seed42': [
        PYTHON, SCRIPTS / 'train_logmel_cnn.py', '--data-dir', DATA_DIR,
        '--models-dir', OUTPUT_DIR / 'separable_headsep_e100' / 'models',
        '--submissions-dir', OUTPUT_DIR / 'separable_headsep_e100' / 'submissions',
        '--experiments-dir', OUTPUT_DIR / 'separable_headsep_e100' / 'experiments',
        '--n-mels', '128', '--frames', '512', '--architecture', 'separable_residual',
        '--activation', 'relu', '--initializer', 'he_normal', '--head-hidden', '256',
        '--head-dropout', '0.3', '--optimizer', 'adam', '--scheduler', 'multistep',
        '--lr-milestones', '27,36,43,49,52', '--epochs', '100', '--batch-size', '24',
        '--seed', str(SEED), '--full-train',
    ],
    'globalmel_sep_temporal_e100_seed42': [
        PYTHON, SCRIPTS / 'train_logmel_cnn.py', '--data-dir', DATA_DIR,
        '--models-dir', OUTPUT_DIR / 'globalmel_e100' / 'models',
        '--submissions-dir', OUTPUT_DIR / 'globalmel_e100' / 'submissions',
        '--experiments-dir', OUTPUT_DIR / 'globalmel_e100' / 'experiments',
        '--n-mels', '128', '--frames', '512', '--cache-tag', 'globalmel',
        '--architecture', 'separable_temporal_bigru', '--activation', 'silu',
        '--initializer', 'he_normal', '--head-dropout', '0.3', '--optimizer', 'adamw',
        '--scheduler', 'multistep', '--lr-milestones', '25,39', '--epochs', '100',
        '--batch-size', '24', '--seed', str(SEED), '--full-train',
    ],
    'sep_temporal_f1024_e100_seed42': [
        PYTHON, SCRIPTS / 'train_logmel_cnn.py', '--data-dir', DATA_DIR,
        '--models-dir', OUTPUT_DIR / 'f1024_e100' / 'models',
        '--submissions-dir', OUTPUT_DIR / 'f1024_e100' / 'submissions',
        '--experiments-dir', OUTPUT_DIR / 'f1024_e100' / 'experiments',
        '--n-mels', '128', '--frames', '1024', '--architecture', 'separable_temporal_bigru',
        '--activation', 'silu', '--initializer', 'he_normal', '--head-dropout', '0.3',
        '--optimizer', 'adamw', '--scheduler', 'multistep', '--lr-milestones', '19,25',
        '--epochs', '100', '--batch-size', '12', '--seed', str(SEED), '--full-train',
    ],
}

for name, command in training_commands.items():
    print(name)
    run_command(command, heavy=True)


separable_headsep_e100_seed42
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/train_logmel_cnn.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --models-dir /home/santig14/fing/taa/2_TallerAA/proyecto_actual/codigo/outputs/separable_headsep_e100/models --submissions-dir /home/santig14/fing/taa/2_TallerAA/proyecto_actual/codigo/outputs/separable_headsep_e100/submissions --experiments-dir /home/santig14/fing/taa/2_TallerAA/proyecto_actual/codigo/outputs/separable_headsep_e100/experiments --n-mels 128 --frames 512 --architecture separable_residual --activation relu --initializer he_normal --head-hidden 256 --head-dropout 0.3 --optimizer adam --scheduler multistep --lr-milestones 27,36,43,49,52 --epochs 100 --batch-size 24 --seed 42 --full-train
SKIP: RUN_HEAVY_STEPS=False
globalmel_sep_temporal_e100_seed42
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/train_

## Seleccion de predicciones

En modo liviano se usan los CSV oficiales generados en la corrida `parallel100_20260702`. En modo pesado se toman las predicciones recien entrenadas bajo `proyecto_actual/codigo/outputs/`.


In [7]:
selected_component_paths = {
    component['short']: component['heavy_csv'] if RUN_HEAVY_STEPS else component['official_csv']
    for component in COMPONENTS
}

path_summary = []
for component in COMPONENTS:
    path = selected_component_paths[component['short']]
    path_summary.append({
        'component': component['id'],
        'weight': component['weight'],
        'source': 'heavy run' if RUN_HEAVY_STEPS else 'official archive',
        'path': str(path.relative_to(ROOT)),
        'exists': path.exists(),
    })
    if not path.exists():
        raise FileNotFoundError(path)

path_summary = pd.DataFrame(path_summary)
display(path_summary)


,component,weight,source,path,exists
0,separable_headsep_e100_seed42,0.250,official archive,investigation/submissions/parallel100_20260702...,True
1,globalmel_sep_temporal_e100_seed42,0.375,official archive,investigation/submissions/parallel100_20260702...,True
2,sep_temporal_f1024_e100_seed42,0.375,official archive,investigation/submissions/parallel100_20260702...,True


## Blend final y validacion

El blend promedia probabilidades clase a clase con pesos `0.25`, `0.375` y `0.375`. Luego se valida que el CSV tenga el mismo formato que `sample_submission.csv`, probabilidades en `[0, 1]`, y el hash esperado para la entrega oficial.


In [8]:
blend_args = [PYTHON, SCRIPTS / 'blend_submissions.py']
for component in COMPONENTS:
    blend_args.extend(['--input', selected_component_paths[component['short']], '--weight', str(component['weight'])])
blend_args.extend(['--output', FINAL_SUBMISSION])

run_command(blend_args)

final_df = pd.read_csv(FINAL_SUBMISSION)
assert list(final_df.columns) == list(sample_submission.columns)
assert final_df['fname'].equals(sample_submission['fname'])
assert final_df[label_columns].ge(0.0).all().all()
assert final_df[label_columns].le(1.0).all().all()
assert len(final_df) == EXPECTED_ROWS
assert len(final_df.columns) == EXPECTED_COLUMNS

official_reference = ROOT / 'investigation/results/submissions/parallel100_20260702/e100_headsep25_globalmel375_f1024_375.csv'
if official_reference.exists() and not RUN_HEAVY_STEPS:
    assert sha256(FINAL_SUBMISSION) == sha256(official_reference)

validation = {
    'submission': str(FINAL_SUBMISSION.relative_to(ROOT)),
    'rows': int(len(final_df)),
    'columns': int(len(final_df.columns)),
    'min_probability': float(final_df[label_columns].min().min()),
    'max_probability': float(final_df[label_columns].max().max()),
    'sha256': sha256(FINAL_SUBMISSION),
    'matches_expected_sha': sha256(FINAL_SUBMISSION) == EXPECTED_SHA256,
    'expected_private_lb': EXPECTED_PRIVATE_LB,
}
if not RUN_HEAVY_STEPS:
    assert validation['matches_expected_sha']
display(pd.DataFrame([validation]))


$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/investigation/scripts/blend_submissions.py --input /home/santig14/fing/taa/2_TallerAA/investigation/submissions/parallel100_20260702_separable_headsep_e100_seed42/small_logmel_cnn.csv --weight 0.25 --input /home/santig14/fing/taa/2_TallerAA/investigation/submissions/parallel100_20260702_globalmel_sep_temporal_e100_seed42/small_logmel_cnn.csv --weight 0.375 --input /home/santig14/fing/taa/2_TallerAA/investigation/submissions/parallel100_20260702_sep_temporal_f1024_e100_seed42/small_logmel_cnn.csv --weight 0.375 --output /home/santig14/fing/taa/2_TallerAA/proyecto_actual/codigo/submission.csv


wrote /home/santig14/fing/taa/2_TallerAA/proyecto_actual/codigo/submission.csv from 3 submissions


,submission,rows,columns,min_probability,max_probability,sha256,matches_expected_sha,expected_private_lb
0,proyecto_actual/codigo/submission.csv,3361,81,3.113838e-11,0.999999,4247ab9ff6398fbb1b6af223629d004265e27bb6cbccab...,True,0.67126


## Revisi?n final

Se muestran las primeras filas del CSV generado y un resumen corto de las probabilidades. La entrega oficial queda asociada al private LB `0.67126`; el mejor experimento `0.67674` se deja documentado aparte porque usa bagging interno adicional.


In [9]:
metadata = {
    'config': config_summary,
    'dataset_summary': dataset_summary.to_dict(orient='records'),
    'components': component_table.drop(columns=['official_exists']).to_dict(orient='records'),
    'selected_component_paths': path_summary.to_dict(orient='records'),
    'validation': validation,
    'notes': {
        'final_model': '3-way e100 weighted',
        'official_private_lb': EXPECTED_PRIVATE_LB,
        'best_experimental_private_lb': 0.67674,
        'why_not_best_experimental': 'usa bagging interno adicional en dos ramas temporales',
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

display(final_df.head())
display(final_df[label_columns].agg(['mean', 'std', 'min', 'max']).T.sort_values('mean', ascending=False).head(10))
print(f'submission_path: {FINAL_SUBMISSION}')
print(f'metadata_path: {METADATA_PATH}')
print(f'sha256: {validation["sha256"]}')


,fname,Accelerating_and_revving_and_vroom,Accordion,Acoustic_guitar,Applause,Bark,Bass_drum,Bass_guitar,Bathtub_(filling_or_washing),Bicycle_bell,...,Toilet_flush,Traffic_noise_and_roadway_noise,Trickle_and_dribble,Walk_and_footsteps,Water_tap_and_faucet,Waves_and_surf,Whispering,Writing,Yell,Zipper_(clothing)
0,4260ebea.wav,1.122018e-07,4.574525e-06,5.505015e-05,2.008303e-05,0.000025,4.799309e-07,0.000001,2.808481e-01,0.000114,...,1.063485e-05,3.814391e-07,3.207197e-02,3.848369e-05,1.825915e-01,5.841105e-05,0.000044,1.658726e-03,0.000007,0.000340
1,426eb1e0.wav,1.839453e-05,8.800063e-06,7.804904e-08,9.998058e-01,0.000003,6.426198e-05,0.000002,7.938651e-04,0.000146,...,9.723566e-07,5.187078e-04,2.823301e-04,1.329407e-03,3.181783e-06,3.435447e-05,0.000002,5.123699e-09,0.000306,0.000036
2,428d70bb.wav,3.342702e-05,1.482364e-05,5.692660e-07,7.526647e-08,0.000059,6.115480e-05,0.000022,3.104282e-06,0.000032,...,3.277350e-06,2.218264e-04,3.582776e-06,3.784563e-06,1.241312e-07,5.924637e-05,0.003608,6.831209e-05,0.000176,0.001556
3,4292b1c9.wav,3.484721e-04,1.685369e-06,1.209903e-05,1.641163e-06,0.000005,2.489413e-07,0.000022,4.402609e-07,0.000005,...,2.681818e-05,1.529014e-07,1.619968e-04,4.583140e-08,1.864286e-04,3.944828e-09,0.000024,5.313329e-06,0.000076,0.000610
4,429c5071.wav,3.528562e-05,2.259558e-07,4.736396e-04,3.093320e-06,0.007229,1.287479e-04,0.000147,1.188675e-05,0.000002,...,4.202741e-04,1.812035e-06,5.218857e-08,2.902128e-03,8.119757e-06,1.712808e-03,0.000004,1.210801e-02,0.000016,0.010544


,mean,std,min,max
Crowd,0.031166,0.151425,3.773762e-09,0.999800
Accelerating_and_revving_and_vroom,0.030812,0.130888,1.086747e-09,0.998097
Stream,0.029286,0.143739,4.559610e-10,0.999944
Motorcycle,0.029081,0.125219,2.399285e-09,0.999843
Waves_and_surf,0.028673,0.122891,5.036096e-10,0.998739
Zipper_(clothing),0.027466,0.135553,2.865637e-09,0.999982
Applause,0.027109,0.148925,2.465017e-09,0.999991
Cheering,0.026756,0.146660,4.738633e-09,0.999960
Traffic_noise_and_roadway_noise,0.025604,0.111240,2.795600e-09,0.993994
Water_tap_and_faucet,0.025257,0.117323,5.617477e-10,0.997940


submission_path: /home/santig14/fing/taa/2_TallerAA/proyecto_actual/codigo/submission.csv
metadata_path: /home/santig14/fing/taa/2_TallerAA/proyecto_actual/codigo/pipeline_final_taller_2_metadata.json
sha256: 4247ab9ff6398fbb1b6af223629d004265e27bb6cbccabf53ec4969a96c61cab
